<a href="https://colab.research.google.com/github/temari1013/Policy_Optimization_Tutorial/blob/main/ex2_actor_critic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 課題内容

ここでは，`2_policy_graident.ipynb`および`3_actor_critic.ipynb`で学んだ内容を踏まえて，以下の課題について実験的に評価します．


## 課題１．バッチ型Actor-Critic法とオンライン型Actor-Critic法の比較

`3_actor_critic.ipynb`にて紹介した２つのActor-Critic法（オンライン更新型，バッチ更新型）を，２つ以上の環境（一つは`LunarLander-v2`とする）において比較しましょう．
公平な比較となるように，方策は２つの方法で同じモデルとする．
また，必ずしも公平とは言えないが，学習率パラメータについてはノートブックに示した値を用いることにする．

強化学習では累積報酬の期待値の最大化が目的となるため，方策の評価には平均累積報酬値を用いることとする（`ex1_evolutionary_policy_optimization.ipynb`とは異なることにに注意）．
また，`ex1_evolutionary_policy_optimization.ipynb`と同様，各方法で乱数を変えて5試行学習を行うこととし，最大ステップ数を$10^6$とした場合の学習曲線を作図すること．
ただし，一度の学習にそれなりの時間を必要とするため，所望の図が作図できるのかの確認をする際には，$10^4$ステップなどと少なめに設定して動作検証をすることが望ましい．
また，`Pendulum-v1`と異なり，`LunarLander-v2`の一エピソード毎のステップ数は異なるため，図の横軸は公平な比較となるように，ステップ数とすべきであることに注意しましょう．




## 課題２．オンライン型Actor-Critic法における方策表現能力の影響

オンライン型Actor-Critic法において，ActorおよびCriticの中間ノード数（`dim_hidden`）を変更し，その影響を調査する．
中間ノード数は，32, 64, 128, 256 とし，ActorとCriticそれぞれについて変更した場合の影響を調べること．
それ以外の設定については課題１と共通とする．

# レポートについて

レポートの執筆方法は`ex1_evolutionary_policy_optimization.ipynb`に従ってください．ただし，実験目的，設定，結果は課題毎に異なるため，課題１，２のそれぞれについて，実験目的から実験結果までを記載してください．共通の設定については，例えば課題２についてまとめている箇所において，「〜については課題１と同様とする．」などと記載すれば十分です．異なる点については必ず明記しましょう．

## バッチ型のLunarLander

In [ ]:
!pip install swig
!apt install -y xvfb
!pip install pyvirtualdisplay
!pip install gymnasium[box2d]

In [ ]:
import random
import torch
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
import seaborn as sns
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Variable
from torch.distributions import Categorical
from IPython import display
from pyvirtualdisplay import Display


# gpuが使用される場合の設定
device = torch.device("cuda" if torch.cuda.is_available() else "cpu" )

In [ ]:
def rollout(envname, policy=None, render=False, seed=None):
    if render:
        env = gym.make(envname, render_mode="rgb_array")
    else:
        env = gym.make(envname)
    history = []
    img = []

    # 乱数の設定
    if seed is not None:
        random.seed(int(seed))
    envseed = random.randint(0, 1000)
    actseed = random.randint(0, 1000)
    observation, info = env.reset(seed=envseed)
    env.action_space.seed(actseed)

    # 可視化用の設定
    if render:
        d = Display()
        d.start()
        img.append(env.render())

    # メインループ（環境とのインタラクション）
    terminated = False
    truncated = False
    while not (terminated or truncated):

        # 行動を選択
        if policy is None:
            action = env.action_space.sample()
        else:
            action = policy(observation)

        # 行動を実行
        next_observation, reward, terminated, truncated, info = env.step(action)
        history.append([observation, action, next_observation, reward, terminated, truncated, info])
        observation = next_observation
        if render:
            display.clear_output(wait=True)
            img.append(env.render())
    env.close()
    return history, img


def visualize(img):
    dpi = 72
    interval = 50
    plt.figure(figsize=(img[0].shape[1]/dpi, img[0].shape[0]/dpi), dpi=dpi)
    patch = plt.imshow(img[0])
    plt.axis=('off')
    animate = lambda i: patch.set_data(img[i])
    ani = animation.FuncAnimation(plt.gcf(), animate, frames=len(img), interval=interval)
    display.display(display.HTML(ani.to_jshtml()))
    plt.close()


def cumulative_reward(history):
    return sum(hist[3] for hist in history)

In [ ]:
class Actor(nn.Module):
    def __init__(self, dim_state, num_action, dim_hidden=128):
        super(Actor, self).__init__()
        self.fc1 = nn.Linear(dim_state, dim_hidden)
        self.fc2 = nn.Linear(dim_hidden, num_action)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.softmax(self.fc2(x), dim=0)
        return x

In [ ]:
class Critic(nn.Module):
    def __init__(self, dim_state, dim_hidden=128):
        super(Critic, self).__init__()
        self.fc1 = nn.Linear(dim_state, dim_hidden)
        self.fc2 = nn.Linear(dim_hidden, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
class BatchActorCriticAgent:
    def __init__(self, env, actor, critic, device, lr_a, lr_c):
        self.device = device
        self.actor = actor
        self.critic = critic
        self.env = env
        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=lr_a, betas=(0.9, 0.999))
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=lr_c, betas=(0.9, 0.999))

    def __call__(self, observation):
        return self.select_action(observation)[0]

    def select_action(self, observation):
        # 行動選択
        observation_ = Variable(torch.Tensor(observation)).to(self.device)
        action_probs = self.actor(observation_)
        log_probs = action_probs.log()
        action = Categorical(action_probs).sample()
        return action.data.cpu().numpy(), log_probs[action]

    def rollout_with_update(self):
        # 1 エピソード実行
        observation, info = self.env.reset()
        steps = 0
        l_observation = []
        l_next_observation = []
        l_reward = []
        l_terminated = []
        l_log_prob = []
        terminated = False
        truncated = False
        # エピソード
        while not (terminated or truncated):
            action, log_prob = self.select_action(observation)
            next_observation, reward, terminated, truncated, info = self.env.step(action)
            l_observation.append(observation)
            l_next_observation.append(next_observation)
            l_reward.append(reward)
            l_terminated.append(terminated)
            l_log_prob.append(log_prob)
            observation = next_observation
            steps +=1
        # 更新
        self.update(l_observation, l_next_observation, l_reward, l_terminated, l_log_prob)
        return l_reward

    def update(self, observation, next_observation, reward, terminated, log_prob):
        obs_tensor = torch.FloatTensor(np.array(observation)).to(self.device)
        next_obs_tensor = torch.FloatTensor(np.array(next_observation)).to(self.device)
        reward_tensor = torch.FloatTensor(np.array(reward)).reshape((-1, 1)).to(self.device)
        flg_tensor = torch.FloatTensor(np.array(terminated)).reshape((-1, 1)).to(self.device)

        vtt = (reward_tensor + (1 - flg_tensor) * self.critic(next_obs_tensor)).detach()
        vt = self.critic(obs_tensor)

        # Actor の更新
        loss_a = - sum([delta * lp for delta, lp in zip(vtt - vt.detach(), log_prob)]) / len(reward)
        self.actor_optimizer.zero_grad()
        loss_a.backward()
        self.actor_optimizer.step()

        # Critic の更新
        loss_c = torch.sum((vtt - vt)**2) / len(reward)
        self.critic_optimizer.zero_grad()
        loss_c.backward()
        self.critic_optimizer.step()

        return loss_a, loss_c

In [ ]:
# 課題文は LunarLander-v2 を指定しているが、この Colab 環境の Gymnasium では使用できないため v3 で実験する。
envname = "LunarLander-v3"
dim_state = 8
num_action = 4
env = gym.make(envname)

actor = Actor(dim_state = 8, num_action = 4).to(device)
critic = Critic(dim_state = 8).to(device)
agent = BatchActorCriticAgent(env, actor, critic, device, lr_a=2e-4, lr_c=2e-3)

In [ ]:
max_steps = 10**4
max_episodes = 10000
interval = 100
returns = []
cumulative_steps = []
total_steps = 0

for episode in range(max_episodes):
    if total_steps >= max_steps:
        break
    rewards = agent.rollout_with_update()
    total_steps += len(rewards)
    returns.append(np.sum(rewards))
    cumulative_steps.append(total_steps)
    if (episode + 1) % interval == 0:
        print(episode + 1, total_steps, np.mean(returns[-interval:]), np.std(returns[-interval:]))

if returns and len(returns) % interval != 0:
    print(len(returns), total_steps, np.mean(returns[-(len(returns) % interval):]), np.std(returns[-(len(returns) % interval):]))

excess_steps = max(0, total_steps - max_steps)
print("total_steps:", total_steps, "excess_steps:", excess_steps)

In [ ]:
return_array = np.zeros(50)
for i in range(len(return_array)):
    history, img = rollout(envname, policy=agent, render=False)
    return_array[i] = cumulative_reward(history)

fig, ax = plt.subplots()
sns.ecdfplot(data=-return_array, ax=ax)
ax.set_xlim(-400, 400)
plt.grid()

In [ ]:
history, img = rollout(envname, policy=agent, render=True)
print(cumulative_reward(history))
visualize(img)

## オンライン型のLunarLander

In [ ]:
class OnlineActorCriticAgent:
    def __init__(self, env, actor, critic, device, lr_a, lr_c):
        self.device = device
        self.actor = actor
        self.critic = critic
        self.env = env
        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=lr_a, betas=(0.9, 0.999))
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=lr_c, betas=(0.9, 0.999))

    def __call__(self, observation):
        return self.select_action(observation)[0]

    def select_action(self, observation):
        observation_ = Variable(torch.Tensor(observation)).to(self.device)
        action_probs = self.actor(observation_)
        log_probs = action_probs.log()
        action = Categorical(action_probs).sample()
        return action.data.cpu().numpy(), log_probs[action]

    def rollout_with_update(self):
        observation, info = self.env.reset()
        rewards = []
        terminated = False
        truncated = False
        while not (terminated or truncated):
            action, log_prob = self.select_action(observation)
            next_observation, reward, terminated, truncated, info = self.env.step(action)
            self.update(observation, next_observation, reward, terminated, log_prob)
            rewards.append(reward)
            observation = next_observation
        return rewards

    def update(self, observation, next_observation, reward, terminated, log_prob):
        if terminated:
            vtt = reward
        else:
            vtt = reward + self.critic(torch.Tensor(next_observation).to(self.device)).detach()
        vt = self.critic(torch.Tensor(observation).to(self.device))

        loss_a = -sum((vtt - vt.detach()) * log_prob)
        self.actor_optimizer.zero_grad()
        loss_a.backward()
        self.actor_optimizer.step()

        loss_c = sum((vtt - vt)**2)
        self.critic_optimizer.zero_grad()
        loss_c.backward()
        self.critic_optimizer.step()

        return loss_a, loss_c

In [ ]:
online_env = gym.make(envname)
online_actor = Actor(dim_state=dim_state, num_action=num_action).to(device)
online_critic = Critic(dim_state=dim_state).to(device)
online_agent = OnlineActorCriticAgent(
    online_env, online_actor, online_critic, device, lr_a=2e-6, lr_c=2e-5
)

In [ ]:
online_max_steps = 10**4
online_max_episodes = 10000
online_interval = 100
online_returns = []
online_cumulative_steps = []
online_total_steps = 0

for episode in range(online_max_episodes):
    if online_total_steps >= online_max_steps:
        break
    rewards = online_agent.rollout_with_update()
    online_total_steps += len(rewards)
    online_returns.append(np.sum(rewards))
    online_cumulative_steps.append(online_total_steps)
    if (episode + 1) % online_interval == 0:
        print(episode + 1, online_total_steps, np.mean(online_returns[-online_interval:]), np.std(online_returns[-online_interval:]))

if online_returns and len(online_returns) % online_interval != 0:
    last_interval = len(online_returns) % online_interval
    print(len(online_returns), online_total_steps, np.mean(online_returns[-last_interval:]), np.std(online_returns[-last_interval:]))

online_excess_steps = max(0, online_total_steps - online_max_steps)
print("total_steps:", online_total_steps, "excess_steps:", online_excess_steps)